# Test enakovrednosti vezij

Preveri, ali optimizirana vezja (`qiskit_only`, `pyzx_only`, `qiskit_then_pyzx`, `pyzx_then_qiskit`) delujejo enako kot izvirno vezje (`original.qasm`).
Vezja so shranjena v `results/<zagon>/<qubits_N | depth_D>/circuit_XX/*.qasm` (glej `main.ipynb`, `qasm2.dump`).

Primerjava je do globalne faze natančno (ta fizikalno ni opazljiva):
- do `MAX_UNITARY_QUBITS` (6) kubitov: primerjava polnih unitarnih matrik (`Operator.equiv`), kar je popoln dokaz enakosti,
- do `MAX_STATEVECTOR_QUBITS` (15) kubitov: simulacija na naključnih vhodnih stanjih (verjetnostni test, ujame napake z verjetnostjo ~1),
- nad tem: preskočeno (pomnilnik 2^n).

In [ ]:
import os
import time

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

results_root = "results"
run_dir = None            # None = zadnji zagon v results_root
MAX_UNITARY_QUBITS = 6        # Operator je počasen (10 kubitov: ~1-2 min na par)
MAX_STATEVECTOR_QUBITS = 15   # 20 kubitov: ~minute na evolucijo; dvigni po potrebi
N_RANDOM_STATES = 1           # ena naključna stanje zadošča (napaka jo zgreši z verjetnostjo ~0)
ATOL = 1e-8
rng = np.random.default_rng(0)

optimized_files = ["qiskit_only", "pyzx_only", "qiskit_then_pyzx", "pyzx_then_qiskit"]

if run_dir is None:
    run_dir = os.path.join(results_root, sorted(os.listdir(results_root))[-1])
print("Zagon:", run_dir)

Zagon: results\2026-10-31_10-43-22


In [ ]:
def equal_unitary(a: QuantumCircuit, b: QuantumCircuit) -> bool:
    return Operator(a).equiv(Operator(b), atol=ATOL)


def random_inputs(n: int) -> list:
    states = []
    for _ in range(N_RANDOM_STATES):
        psi = rng.normal(size=2 ** n) + 1j * rng.normal(size=2 ** n)
        states.append(psi / np.linalg.norm(psi))
    return states


def equal_random_states(a_out: list, b: QuantumCircuit, inputs: list) -> bool:
    # |<a|b>| = 1 <=> enaka stanja do globalne faze; a_out = izhodna stanja izvirnega vezja (izračunana enkrat)
    return all(abs(np.vdot(sa, Statevector(psi).evolve(b).data)) > 1 - ATOL
               for sa, psi in zip(a_out, inputs))


def check_pair(original: QuantumCircuit, optimized: QuantumCircuit, ref: list, inputs: list):
    if original.num_qubits != optimized.num_qubits:
        return False, "število kubitov"
    if original.num_qubits <= MAX_UNITARY_QUBITS:
        return equal_unitary(original, optimized), "unitarna matrika"
    return equal_random_states(ref, optimized, inputs), "naključna stanja"


def circuit_dirs(run_dir: str):
    for group in sorted(os.listdir(run_dir)):
        group_path = os.path.join(run_dir, group)
        if os.path.isdir(group_path):
            for circ in sorted(os.listdir(group_path)):
                yield group, circ, os.path.join(group_path, circ)

In [4]:
rows = []
start = time.perf_counter()
for group, circ, path in circuit_dirs(run_dir):
    original = QuantumCircuit.from_qasm_file(os.path.join(path, "original.qasm"))
    if original.num_qubits > MAX_STATEVECTOR_QUBITS:
        for name in optimized_files:
            rows.append({"group": group, "circuit": circ, "method": name, "qubits": original.num_qubits,
                         "equal": None, "check": "preskočeno"})
        continue
    n = original.num_qubits
    inputs = random_inputs(n) if MAX_UNITARY_QUBITS < n else []
    ref = [Statevector(psi).evolve(original).data for psi in inputs]
    for name in optimized_files:
        optimized = QuantumCircuit.from_qasm_file(os.path.join(path, f"{name}.qasm"))
        equal, how = check_pair(original, optimized, ref, inputs)
        rows.append({"group": group, "circuit": circ, "method": name, "qubits": original.num_qubits,
                     "equal": equal, "check": how})
    if circ.endswith("_09"):
        print(f"{group} končano ({time.perf_counter() - start:.0f} s)")
print(f"Preverjeno {len(rows)} parov vezij v {time.perf_counter() - start:.0f} s")

depth_100 končano (14 s)
depth_150 končano (30 s)
depth_200 končano (49 s)
depth_25 končano (53 s)
depth_50 končano (59 s)
qubits_10 končano (70 s)
qubits_15 končano (189 s)
qubits_5 končano (196 s)
Preverjeno 400 parov vezij v 196 s


In [5]:
from collections import defaultdict

stats = defaultdict(lambda: {"ok": 0, "fail": 0, "skipped": 0})
for r in rows:
    key = (r["group"], r["method"])
    stats[key]["skipped" if r["equal"] is None else "ok" if r["equal"] else "fail"] += 1

print(f"{'skupina':<12}{'metoda':<20}{'enaka':>7}{'napačna':>9}{'preskočena':>12}")
for (group, method), s in sorted(stats.items()):
    print(f"{group:<12}{method:<20}{s['ok']:>7}{s['fail']:>9}{s['skipped']:>12}")

failed = [r for r in rows if r["equal"] is False]
print()
if failed:
    print(f"NAPAKA: {len(failed)} vezij ni enakovredna izvirnemu vezju:")
    for r in failed:
        print("  ", r["group"], r["circuit"], r["method"])
else:
    print("Vsa preverjena vezja so enakovredna izvirnemu vezju (do globalne faze).")
assert not failed

skupina     metoda                enaka  napačna  preskočena
depth_100   pyzx_only                10        0           0
depth_100   pyzx_then_qiskit         10        0           0
depth_100   qiskit_only              10        0           0
depth_100   qiskit_then_pyzx         10        0           0
depth_150   pyzx_only                10        0           0
depth_150   pyzx_then_qiskit         10        0           0
depth_150   qiskit_only              10        0           0
depth_150   qiskit_then_pyzx         10        0           0
depth_200   pyzx_only                10        0           0
depth_200   pyzx_then_qiskit         10        0           0
depth_200   qiskit_only              10        0           0
depth_200   qiskit_then_pyzx         10        0           0
depth_25    pyzx_only                10        0           0
depth_25    pyzx_then_qiskit         10        0           0
depth_25    qiskit_only              10        0           0
depth_25    qiskit_then_